# 02 - Leakage-Safe Preprocessing

This notebook runs the real preprocessing pipeline on a reduced synthetic sample and checks the central rule: **split consumers first, then fit preprocessing on training consumers only**. It also shows the 133 model features.

In [ ]:
from pathlib import Path
import json
import sys

# Headless-safe plotting; Jupyter still renders figures inline.
import matplotlib
matplotlib.use("Agg")

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT))
print("Project root:", ROOT)


In [ ]:
from preprocessing.pipeline import prepare_data
from utilities.config import load_config

cfg = load_config("quick", [
    "data.synthetic.consumers=40",
    "data.synthetic.days=60",
    "data.synthetic.theft_rate=0.16",
])
splits = prepare_data(cfg, seed=42, log=print)
print("Source:", splits.source, "| synthetic:", splits.is_synthetic)
print(json.dumps(splits.preprocessing_report, indent=2))


## Split sizes and feature count

In [ ]:
print("Consumers train/val/test:",
      len(splits.train_consumers), len(splits.val_consumers), len(splits.test_consumers))
print("Windows train/val/test:",
      len(splits.train), len(splits.val), len(splits.test))
print("Features:", len(splits.train.feature_names))
print("First features:", splits.train.feature_names[:12])
print("Train tensor shape:", splits.train.X_matrix.shape)
print("Val tensor shape:  ", splits.val.X_matrix.shape)


## Leakage check: consumer sets must be disjoint

In [ ]:
train_ids = set(map(str, splits.train_consumers))
val_ids = set(map(str, splits.val_consumers))
test_ids = set(map(str, splits.test_consumers))
assert train_ids.isdisjoint(val_ids)
assert train_ids.isdisjoint(test_ids)
assert val_ids.isdisjoint(test_ids)
print("PASS: train/validation/test consumer sets are disjoint")
print("Train theft-window ratio:", round(splits.train.theft_ratio, 4))
print("Val theft-window ratio:  ", round(splits.val.theft_ratio, 4))
print("Test theft-window ratio: ", round(splits.test.theft_ratio, 4))


## Cleaning and imbalance reports

In [ ]:
print(json.dumps(splits.cleaning_report, indent=2))
print(json.dumps(splits.imbalance, indent=2))


The full report also stores `split_unit=consumer` and train-fitted scaler/outlier metadata in `results/reports/final_experiment_report.md`.